<a href="https://colab.research.google.com/github/winter-pro/Statistical-Learning-e23091/blob/main/Assignment_7c_Item_Response_Prediction_and_Click_Through_Rate_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 1: Bayesian Estimation of a User Ability Parameter
Markdown: The 2PL Item Response Probability Model

Assuming that the item responses are conditionally independent given $\Theta=\theta$, the likelihood is

$$P(Y=y\mid \Theta=\theta)=\prod_{i=1}^n \left[ p_i(\theta) \right]^{y_i} \left[ 1-p_i(\theta) \right]^{1-y_i},$$

where

$$p_i(\theta) =\frac{1}{1+e^{-a_i(\theta-b_i)}}.$$

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range of latent ability values (theta)
theta_vals = np.linspace(-6, 6, 300)

# Define configurations to plot
curves = [
    {"a": 0.5, "b": 0, "line_style": "dash"},
    {"a": 1.5, "b": -2, "line_style": "solid"},
    {"a": 1.5, "b": 0, "line_style": "solid"},
    {"a": 1.5, "b": 2, "line_style": "solid"},
]

fig = go.Figure()

for curve in curves:
    a = curve["a"]
    b = curve["b"]
    style = curve["line_style"]
    p_vals = p_i(theta_vals, a, b)
    fig.add_trace(go.Scatter(
        x=theta_vals, y=p_vals, mode='lines',
        name=f"a = {a}, b = {b}", line=dict(dash=style, width=2.5)
    ))

fig.update_layout(
    title={'text': "Two-Parameter Logistic (2PL) Item Response Curves", 'y':0.9, 'x':0.5, 'xanchor': 'center', 'yanchor': 'top'},
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.05, bgcolor="rgba(255,255,255,0.8)")
)
fig.show()

Markdown: Posterior Distribution

At step $k$ (for $k = 1, \dots, n$), the current item response is $y_k$, and its likelihood contribution is:

$$L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$

The sequential posterior density of $\Theta$ given the running response history vector $\mathbf{y}^{(k)}$ is updated recursively as follows:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) = \frac{ \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}) }{ \int_{\mathbb R} \left[ p_k(s)^{y_k} (1 - p_k(s))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(s\mid\mathbf{y}^{(k-1)})\,ds }$$

**Definition of Terms:**
* $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})$ is the **prior density** for the current step (which is the posterior inherited from the previous step).
* For the very first item ($k=1$), the base prior is the initialized distribution:
$$f_{\Theta\mid\mathbf{Y}^{(0)}}(\theta\mid\mathbf{y}^{(0)}) = f_\Theta^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}}\exp\left(-\frac{\theta^2}{2}\right)$$
* The denominator is the **local normalizing constant**, integrating out $\theta$ across the current likelihood-prior product to ensure the area under the new step-$k$ density curve integrates to 1.

Markdown: Baye's Estimate and the MAP Estimate

Under a sequential framework, point estimates like the **Posterior Mean** (Bayes estimate under squared-error loss) or the **Maximum A Posteriori (MAP)** estimate are updated dynamically at each step $k$ using the running posterior density $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.

1. Running Posterior Mean (Bayes Estimate)
The Bayesian estimate of the user's latent ability at step $k$, minimizing the expected squared-error loss, is the expected value of the current posterior distribution:
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb E[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \int_{\mathbb R} \theta \, f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})\,d\theta$$

2. Running Maximum A Posteriori (MAP) Estimate
Alternatively, the most likely ability parameter at step $k$ corresponds to the mode (peak) of the current posterior density curve:
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} \in \arg\max_{\theta\in\mathbb R} f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$$

**Computation Note**
In practice, as the user answers more items, the computational grid tracks the updated array values for $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.
* $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ is updated via a running vector dot-product: `np.trapz(theta * current_posterior, theta)`.
* $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ is extracted simply via the index of the maximum value: `theta[np.argmax(current_posterior)]`.

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# PART 1: SEQUENTIAL BAYESIAN UPDATE (MANUAL 4-ITEM SIMULATION)
# =====================================================================

theta = np.linspace(-5, 5, 500)
prior = stats.norm.pdf(theta, 0, 1)

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

running_items = [
    {"a": 1.0, "b": -1.5, "y": 1},
    {"a": 1.5, "b": 0.5,  "y": 1},
    {"a": 1.2, "b": 1.5,  "y": 0},
    {"a": 2.0, "b": 0.2,  "y": 1}
]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=theta, y=prior, mode='lines', name='Initial Prior: N(0,1)', line=dict(dash='dash', width=2.5, color='gray')))

current_posterior = prior.copy()

for idx, item in enumerate(running_items):
    a, b, y = item["a"], item["b"], item["y"]
    prob = p_i(theta, a, b)
    likelihood = (prob ** y) * ((1 - prob) ** (1 - y))
    current_posterior = current_posterior * likelihood
    current_posterior /= np.trapezoid(current_posterior, theta)

    result_text = "Correct" if y == 1 else "Incorrect"
    fig1.add_trace(go.Scatter(x=theta, y=current_posterior, mode='lines', name=f"Step {idx+1}: After Item {idx+1} ({result_text}, a={a}, b={b})", line=dict(width=2)))

fig1.update_layout(title={'text': "Sequential Bayesian Update of User Ability (θ)", 'y': 0.95, 'x': 0.5, 'xanchor': 'center'}, xaxis_title="Latent Ability Parameter (θ)", yaxis_title="Probability Density f(θ | y)", template="plotly_white", hovermode="x unified")
fig1.show()

# =====================================================================
# PART 2: PERFORMANCE TRACKING & CONVERGENCE TIMELINE (20 ITEMS)
# =====================================================================

np.random.seed(42)
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)

a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))
current_posterior_sim = stats.norm.pdf(theta_grid, 0, 1)

for k in range(n_items):
    a_k, b_k = a_params[k], b_params[k]
    prob_true = p_i(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0
    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    current_posterior_sim = current_posterior_sim * likelihood
    current_posterior_sim /= np.trapezoid(current_posterior_sim, theta_grid)

    running_bayes.append(np.trapezoid(theta_grid * current_posterior_sim, theta_grid))
    running_map.append(theta_grid[np.argmax(current_posterior_sim)])

fig2 = go.Figure()
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red", line_width=2, annotation_text=f"True Ability (θ = {theta_true})", annotation_position="bottom right")
fig2.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines+markers', name='Posterior Mean (θ̂_Bayes)', line=dict(color='blue', width=2.5), marker=dict(size=6)))
fig2.add_trace(go.Scatter(x=steps, y=running_map, mode='lines+markers', name='MAP Estimate (θ̂_MAP)', line=dict(color='green', width=2), marker=dict(size=6, symbol='square')))

fig2.update_layout(title={'text': "Convergence of Latent Ability Estimators (θ) Over Time", 'y': 0.93, 'x': 0.5, 'xanchor': 'center'}, xaxis_title="Sequence / Item Position (k)", yaxis_title="Estimated Ability (θ̂)", template="plotly_white", hovermode="x unified")
fig2.show()

Question 2: Bayesian Tracking of Click-Through Rates (CTR)

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0, 1, 500)
beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative State: Beta(1,1)", "color": "gray", "dash": "dash"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed State: Beta(2,8)", "color": "blue", "dash": "solid"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed State: Beta(8,2)", "color": "green", "dash": "solid"}
]

fig = go.Figure()
for config in beta_configs:
    pdf_vals = stats.beta.pdf(theta_grid, config["alpha"], config["beta"])
    fig.add_trace(go.Scatter(x=theta_grid, y=pdf_vals, mode='lines', name=config["name"], line=dict(color=config["color"], dash=config["dash"], width=2.5)))

fig.update_layout(title={'text': "Structural Variations of the Beta(α, β) Probability Density Function", 'y': 0.93, 'x': 0.5, 'xanchor': 'center'}, xaxis_title="Parameter Value (θ)", yaxis_title="Probability Density f(θ)", xaxis=dict(range=[0, 1]), template="plotly_white", hovermode="x unified")
fig.show()

Markdown: Sequential Likelihood and Posterior Mean

**1. Single Isolated Response Likelihood**
At any specific impression step $k$, the user interaction $y_k \in \{0, 1\}$ is modeled as an independent Bernoulli trial conditional on the true click probability $\theta$.

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

**2. Joint Likelihood Function for the Running History**
The joint history likelihood simplifies compactly to:

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{C_k} (1 - \theta)^{k - C_k}$$

**Closed-Form Analytical Updates (Conjugacy)**
Applying Bayes' Theorem sequentially, the posterior density after observing the newest single event $y_k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} \cdot (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

The new analytical state parameters $(\alpha_k, \beta_k)$ are simple deterministic linear updates:
$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$

**Posterior Mean**
The expected value is given by the standard formula for a Beta distribution's mean:
$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + C_k}{(\alpha_0 + \beta_0) + k}$$

In [4]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)
theta_true = 0.35
n_impressions = 100
steps = list(range(n_impressions + 1))

alpha_param = 1
beta_param = 1
theta_grid = np.linspace(0, 1, 500)
milestones = [0, 1, 2, 5, 10, 30, 50, 100]

running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]

fig1 = go.Figure()
prior_density = stats.beta.pdf(theta_grid, alpha_param, beta_param)
fig1.add_trace(go.Scatter(x=theta_grid, y=prior_density, mode='lines', name='Initial Prior: Beta(1,1)', line=dict(dash='dash', width=2.5, color='gray')))

for k in range(1, n_impressions + 1):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha_param += y_k
    beta_param += (1 - y_k)

    theta_bayes_k = alpha_param / (alpha_param + beta_param)
    if alpha_param > 1 and beta_param > 1:
        theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        theta_map_k = 0.0 if alpha_param <= beta_param else 1.0

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

    if k in milestones:
        density_k = stats.beta.pdf(theta_grid, alpha_param, beta_param)
        result_text = "Click" if y_k == 1 else "No Click"
        fig1.add_trace(go.Scatter(x=theta_grid, y=density_k, mode='lines', name=f"Step {k}: After Event ({result_text}, α={alpha_param}, β={beta_param})", line=dict(width=2)))

fig1.add_vline(x=theta_true, line_dash="dot", line_color="red", line_width=2, annotation_text=f"True CTR ({theta_true})", annotation_position="top right")
fig1.update_layout(title={'text': "Analytical Posterior Density Progression", 'y': 0.95, 'x': 0.5, 'xanchor': 'center'}, xaxis_title="Conversion Rate Parameter (θ)", yaxis_title="Probability Density", template="plotly_white")
fig1.show()

fig2 = go.Figure()
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red", line_width=2, annotation_text=f"True CTR (θ = {theta_true})")
fig2.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines', name='Exact Posterior Mean', line=dict(color='blue', width=2.5)))
fig2.add_trace(go.Scatter(x=steps, y=running_map, mode='lines', name='Exact MAP Estimate', line=dict(color='green', width=1.5, dash='dot')))
fig2.update_layout(title={'text': "Analytical Beta-Binomial Conjugate Update Timeline", 'y': 0.93, 'x': 0.5, 'xanchor': 'center'}, xaxis_title="Number of User Impressions (k)", yaxis_title="Estimated Conversion Rate (θ̂)", template="plotly_white")
fig2.show()

Question 3: Structural Health Monitoring
Markdown: Formulation and Point Estimators

1. Physical Likelihood Formulation
$$Y_k = g(\theta) + \epsilon_k$$
$$L(y_k \mid \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{\left(y_k - g(\theta)\right)^2}{2\sigma^2} \right)$$

2. Sequential Likelihood and Joint History
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \frac{1}{(2\pi\sigma^2)^{k/2}} \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left(y_i - g(\theta)\right)^2 \right)$$

3. Mathematical Formulation of Bounded Recursive Updates
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{\theta_{\min}}^{\theta_{\max}} L(y_k \mid s) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds}$$

4. Running Point Estimators
**Running Posterior Mean:**
$$\widehat{\theta}_{\text{Bayes}}^{(k)} = \int_{\theta_{\min}}^{\theta_{\max}} \theta \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

**Running Maximum A Posteriori (MAP):**
$$\widehat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in [\theta_{\min}, \theta_{\max}]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(24)
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_sensor_readings = 15

theta_grid = np.linspace(0.01, 1.0, 500)
current_posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
current_posterior /= np.trapezoid(current_posterior, theta_grid)
milestones = [0, 1, 2, 5, 10, 15]

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=current_posterior, mode='lines', name='Prior State: Assumed Healthy', line=dict(dash='dash', width=2.5, color='gray')))

for k in range(1, n_sensor_readings + 1):
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    current_posterior = current_posterior * likelihood
    current_posterior /= np.trapezoid(current_posterior, theta_grid)

    if k in milestones:
        fig.add_trace(go.Scatter(x=theta_grid, y=current_posterior, mode='lines', name=f"Step {k}: Post-Sensor (K={y_k:.2f})", line=dict(width=2)))

fig.add_vline(x=theta_true, line_dash="dot", line_color="red", line_width=2.5, annotation_text=f"True State ({theta_true})")
fig.update_layout(title={'text': "Structural Health Monitoring: Bounded Bayesian Updates", 'y': 0.95, 'x': 0.5, 'xanchor': 'center'}, xaxis_title="Remaining Stiffness Efficiency (θ)", yaxis_title="Probability Density", template="plotly_white")
fig.show()

Question 4: Gaussian Mixture Clustering
Markdown: GMM Conditional Updates

**Part 1: Marginal Density**
$$p(x_i) = \sum_{k=1}^{K} \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$
This is called a Gaussian mixture density because it models a complex probability distribution by "mixing" together $K$ distinct Gaussian components weighted by $\phi_k$.

**Part 2: Posterior Cluster Probability**
$$\gamma_{ik} = P(C_i = k \mid X_i=x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$
This illustrates a conditional update: we modulate our prior belief ($\phi_k$) based on how well the location $x_i$ matches the distribution of cluster $k$.

**Part 3: Soft Cluster Assignment**
$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$
The soft cluster assignment is precisely the conditional expectation of the one-hot encoded latent random vector.

**Part 6: Complete-Data Log-Likelihood**
$$\ell_c=\sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i\mid \mu_k,\Sigma_k) \right]$$
If $z_{ik}$ were known, this expression would be easy to maximize because the log-likelihood collapses into independent maximum likelihood tasks for each individual Gaussian component.

**Part 7 & 8: EM Updates**
$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$
The E-step computes the conditional expectation (responsibilities $\gamma_{ik}$). The M-step then updates $\phi_k$, $\mu_k$, and $\Sigma_k$ by treating these responsibilities as fractional membership weights.

In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import kagglehub

# 1. Download Kaggle Dataset
kagglepath = "arjunbhasin2013/ccdata"
path = kagglehub.dataset_download(kagglepath)
df2 = pd.read_csv(path + "/CC GENERAL.csv")

# 2. GMM Segmenter Class
class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(n_components=self.n_components, covariance_type="full", random_state=self.random_state)

    def prepare_data(self, df, feature_cols, test_size=0.2):
        X = df[feature_cols].dropna().values
        X_scaled = self.scaler.fit_transform(X)
        X_train, X_test = train_test_split(X_scaled, test_size=test_size, random_state=self.random_state)
        return X_train, X_test

    def fit(self, X_train):
        self.model.fit(X_train)
        print("GMM Training complete.")
        print(f"Converged: {self.model.converged_}")
        print(f"Iterations taken: {self.model.n_iter_}")

    def evaluate(self, X_test):
        avg_log_likelihood = self.model.score(X_test)
        print(f"\nValidation Performance (Test Set):\nAverage Log-Likelihood: {avg_log_likelihood:.4f}")
        return avg_log_likelihood

    def plot_density_heatmap(self, X_train, feature_names):
        X_orig = self.scaler.inverse_transform(X_train)
        fig = px.density_heatmap(x=X_orig[:, 0], y=X_orig[:, 1], labels={"x": feature_names[0], "y": feature_names[1]}, title="Empirical Training Data Density Heatmap", marginal_x="histogram", marginal_y="histogram")
        fig.update_traces(colorscale="Viridis", selector=dict(type='histogram2d'))
        fig.update_layout(template="plotly_white")
        fig.show()

    def _generate_contour_base(self, X_data):
        x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
        y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid_points = np.c_[xx.ravel(), yy.ravel()]
        responsibilities = self.model.predict_proba(grid_points)
        max_prob = responsibilities.max(axis=1).reshape(xx.shape)
        grid_orig = self.scaler.inverse_transform(grid_points)
        return grid_orig[:, 0].reshape(xx.shape), grid_orig[:, 1].reshape(yy.shape), max_prob

    def plot_training_assignments(self, X_train, feature_names):
        xx_orig, yy_orig, max_prob = self._generate_contour_base(X_train)
        hard_labels = self.model.predict(X_train)
        X_train_orig = self.scaler.inverse_transform(X_train)
        fig = go.Figure()
        fig.add_trace(go.Contour(x=xx_orig[0, :], y=yy_orig[:, 0], z=max_prob, colorscale="Cividis", contours_coloring="heatmap", opacity=0.6))
        for k in range(self.n_components):
            cluster_mask = hard_labels == k
            fig.add_trace(go.Scatter(x=X_train_orig[cluster_mask, 0], y=X_train_orig[cluster_mask, 1], mode="markers", name=f"Train Cluster {k+1}"))
        fig.update_layout(title="GMM Confidence Boundaries (Training)", xaxis_title=feature_names[0], yaxis_title=feature_names[1], template="plotly_white")
        fig.show()

    def plot_soft_assignments(self, X_test, feature_names):
        xx_orig, yy_orig, max_prob = self._generate_contour_base(X_test)
        hard_labels = self.model.predict(X_test)
        X_test_orig = self.scaler.inverse_transform(X_test)
        fig = go.Figure()
        fig.add_trace(go.Contour(x=xx_orig[0, :], y=yy_orig[:, 0], z=max_prob, colorscale="Cividis", contours_coloring="heatmap", opacity=0.6))
        for k in range(self.n_components):
            cluster_mask = hard_labels == k
            fig.add_trace(go.Scatter(x=X_test_orig[cluster_mask, 0], y=X_test_orig[cluster_mask, 1], mode="markers", name=f"Test Cluster {k+1}"))
        fig.update_layout(title="GMM Confidence Boundaries (Test)", xaxis_title=feature_names[0], yaxis_title=feature_names[1], template="plotly_white")
        fig.show()

# 3. Execute
if __name__ == "__main__":
    features = ["PURCHASES", "CREDIT_LIMIT"]
    segmenter = GMMFinancialSegmenter(n_components=3)
    X_train, X_test = segmenter.prepare_data(df2, features)
    segmenter.fit(X_train)
    segmenter.evaluate(X_test)
    segmenter.plot_density_heatmap(X_train, features)
    segmenter.plot_training_assignments(X_train, features)
    segmenter.plot_soft_assignments(X_test, features)

100%|██████████| 340k/340k [00:00<00:00, 948kB/s]

Extracting files...


GMM Training complete.
Converged: True
Iterations taken: 19

Validation Performance (Test Set):
Average Log-Likelihood: -1.6465
